# 2008 Formula 1 Championship Analysis

This notebook analyzes driver and constructor performance from a CSV file containing 2008 Formula 1 race results. It calculates championship points, individual driver statistics, driver standings, and constructor standings.

In [ ]:
import csv
import io
from google.colab import files


def load_data():
    """Upload and parse the CSV file containing the championship results.

    Returns:
        list[dict]: One dictionary per driver's result in a Grand Prix.
    """
    print("Select and upload 'formula1_data.csv'.")
    uploaded = files.upload()

    file_name = list(uploaded.keys())[0]
    csv_data = uploaded[file_name].decode("utf-8")
    file_buffer = io.StringIO(csv_data)

    championship_data = []
    reader = csv.DictReader(file_buffer)
    for row in reader:
        row["Position"] = int(row["Position"])
        championship_data.append(row)

    return championship_data


def calculate_points(position):
    """Return the points awarded for a finishing position.

    The function uses the 2008 scoring system: 10, 8, 6, 5, 4, 3, 2,
    and 1 point for positions one through eight.
    """
    points_by_position = {1: 10, 2: 8, 3: 6, 4: 5, 5: 4, 6: 3, 7: 2, 8: 1}
    return points_by_position.get(position, 0)


def analyze_driver_performance(driver_name, championship_data):
    """Calculate total points, wins, and podiums for one driver."""
    total_points = 0
    wins = 0
    podiums = 0
    driver_found = False

    for race_result in championship_data:
        if race_result["Driver"].lower() == driver_name.lower():
            driver_found = True
            position = race_result["Position"]
            total_points += calculate_points(position)

            if position == 1:
                wins += 1
            if 1 <= position <= 3:
                podiums += 1

    if not driver_found:
        raise ValueError(f"Driver '{driver_name}' was not found in the dataset.")

    return [total_points, wins, podiums]


def generate_driver_standings(championship_data):
    """Aggregate driver points, sort the standings, and save them to a text file."""
    driver_points = {}

    for race_result in championship_data:
        driver = race_result["Driver"]
        points = calculate_points(race_result["Position"])
        driver_points[driver] = driver_points.get(driver, 0) + points

    ordered_standings = dict(
        sorted(driver_points.items(), key=lambda item: item[1], reverse=True)
    )

    output_file = "drivers_standings_2008.txt"
    with open(output_file, "w", encoding="utf-8") as file:
        file.write("2008 Formula 1 Driver Standings\n")
        for driver, points in ordered_standings.items():
            file.write(f"{driver}: {points}\n")

    print(f"Generated '{output_file}'. Download it from the Colab file panel.")
    return ordered_standings


def generate_constructor_standings(championship_data):
    """Aggregate points by team and return the ordered constructor standings."""
    constructor_points = {}

    for race_result in championship_data:
        team = race_result["Team"]
        points = calculate_points(race_result["Position"])
        constructor_points[team] = constructor_points.get(team, 0) + points

    return dict(
        sorted(constructor_points.items(), key=lambda item: item[1], reverse=True)
    )


if __name__ == "__main__":
    print("Starting the 2008 Formula 1 Championship analysis...\n")

    dataset = load_data()
    print(f"\nLoaded {len(dataset)} rows successfully.\n")

    test_driver = "Hamilton"
    print(f"--- DRIVER ANALYSIS: {test_driver} ---")
    try:
        statistics = analyze_driver_performance(test_driver, dataset)
        print(f"Total points: {statistics[0]}")
        print(f"Wins: {statistics[1]}")
        print(f"Podiums: {statistics[2]}\n")
    except ValueError as error:
        print(error)

    print("--- DRIVER STANDINGS (TOP 5) ---")
    driver_standings = generate_driver_standings(dataset)
    for rank, (driver, points) in enumerate(list(driver_standings.items())[:5], start=1):
        print(f"{rank}. {driver}: {points} points")

    print("\n--- CONSTRUCTOR STANDINGS ---")
    constructor_standings = generate_constructor_standings(dataset)
    for rank, (team, points) in enumerate(constructor_standings.items(), start=1):
        print(f"{rank}. {team}: {points} points")
